# nano Beatrix — the third pass: the curriculum reads, and the address axis

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_pass3.ipynb)

The second pass (nano_frame_pass2.ipynb) settled that the long window is reached by curriculum, not from birth: a copy formed at 1,024 context carries into 4,096 within 4,000 steps, for eight records a frame and for sixteen, where 12,000 steps at 4,096 from birth never form it. This pass runs the reads that follow from that, one run at a time, every finished run shipped as it lands.

The grids, in order. (1) The scrutiny reads by continuation: the second pass's 1,024 scrutiny crafts (nesting 0.6, conflicting definitions 0.3; the question on a nested frame or on a conflicted key) continued 4,000 steps at 4,096, both twins, two seeds, both asks. (2) The sixteen-record frames on the second seeds: a stage-A craft for Pollux seed 2 at 1,024 with eight-record frames, then the Castor and Pollux seed-2 crafts continued at 4,096 on sixteen-record frames. (3) Pollux's hub address axis: the hub is the only window-reaching address in the pair; Pollux at 1,024 with the address at 64 by 32 and at 128 by 64 (the pair's default is 32 by 16), two seeds, then each continued at 4,096 on eight- and sixteen-record frames. If the long-window read moves with the address, scalable windows come from the address.

A starting craft that is not on this machine is fetched from the training repo's bed/weights/ before its run (tokenless). The cells are the second pass's: C1 pins the trainer and holds the decision block; C2 builds the data, prints each grid's plan, runs 200 steps for the price line and probes the push path; C3 is the session; C4 renders the results table. Put a Hugging Face token with write access to the training repo in this notebook's secrets as HF_TOKEN before C1; tokenless, everything runs and nothing ships.


In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.20"
import torch
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| torch', torch.__version__, '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 20), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner ships through the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the record, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the corpus, the records, the weights, runner.log
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # every finished run ships here (its record and its weights); tokenless = nothing ships
CORPUS_BYTES = 160e6                                   # fineweb-edu text for the bed: 160 MB covers 12,000 steps x 8 x 4,096 four times over
BASE = dict(
    arms='castor-both,pollux-both',   # the twins at the decided bank (table + address, 2026-10-12): the best text on both bodies on two seeds
    seeds='1,2',                      # the two-seed rule
    register='caption',               # the first pass read every register (letters, digits, hex); the second pass holds the register still
    ctx=4096, mb=8, steps=12000,      # the recipe of the first pass: 12,000 steps x 8 rows x 4,096
    frames='dyn', plant=0.75,         # three rows in four framed, one in four a plain document: an all-framed diet made a frame specialist of one
                                      # Castor seed that could not read a bare document (the 2026-10-13 probe); the 4090's night of 10-13 reads the diet
    frame_records=4, nest=0.3, overlap=0.0, ask='any', tags='mode',   # the first pass's frame settings: four records a lookup frame, nesting 0.3,
                                      # no conflicts, the question on any frame, MODE + letter tags; each grid below moves one of them
    log_every=100,                    # the trainer's progress line (step, bits, tok/s, time left) feeds the step bar
)
SHORT = dict(BASE, ctx=1024, mb=32)       # stage A: the short window at 32 rows a step, where every copy formed in 12,000 steps (2026-10-13)
CONT = dict(BASE, steps=4000)             # a continuation stage: 4,000 steps at 4,096 x 8 from a stage-A craft (its weights fetched from the Hub if absent)
GRIDS = dict(
    # 1. the scrutiny reads by continuation: the 1,024 scrutiny crafts of 2026-10-13 continued at 4,096 on the scrutiny diet, the ask they were trained on
    scrutiny_cont=dict(CONT, sizes='32,128,256,512', nest=0.6, overlap=0.3, ask='nested,overlap',
                       init='{arm}_s{seed}_L12000_frdyn_cap_mode_sz32-128-256_r4_n60_o30_ask{ask}_p75'),
    # 2a. the missing stage-A craft: Pollux seed 2 with eight-record frames at 1,024 (Castor seed 2 and both seed-1 crafts exist on the Hub)
    stageA_pollux_s2=dict(SHORT, arms='pollux-both', seeds='2', frame_records=8),
    # 2b. sixteen records a frame by continuation on the second seeds (seed 1 read 0.97 / 0.91 / 0.70 / 0.68 / 0.55 for Castor, 0.89 / 0.39 / 0.09 / 0.09 / 0.03 for Pollux)
    capacity16_cont_s2=dict(CONT, seeds='2', frame_records=16, sizes='32,128,256,512,1024',
                            init='{arm}_s{seed}_L12000_frdyn_cap_mode_sz32-128-256_r8_n30_o0_askany_p75'),
    # 3a. Pollux's hub address axis at 1,024: the address at 64 x 32 and 128 x 64 against the pair's 32 x 16 (eight-record frames; the tag carries K and D)
    addr64_stageA=dict(SHORT, arms='pollux-both', frame_records=8, hub_K=64, hub_D=32),
    addr128_stageA=dict(SHORT, arms='pollux-both', frame_records=8, hub_K=128, hub_D=64),
    # 3b. each address continued at 4,096 on eight- and sixteen-record frames to 1,024 bytes
    addr64_cont=dict(CONT, arms='pollux-both', frame_records='8,16', sizes='32,128,256,512,1024', hub_K=64, hub_D=32,
                     init='{arm}_s{seed}_L12000_K{hub_K}D{hub_D}c1_frdyn_cap_mode_sz32-128-256_r8_n30_o0_askany_p75'),
    addr128_cont=dict(CONT, arms='pollux-both', frame_records='8,16', sizes='32,128,256,512,1024', hub_K=128, hub_D=64,
                      init='{arm}_s{seed}_L12000_K{hub_K}D{hub_D}c1_frdyn_cap_mode_sz32-128-256_r8_n30_o0_askany_p75'),
)
SMOKE_STEPS = 200                     # C2's short run on this card for the price line; its push is off
PARALLEL = 1                          # runs at once on the card. ONE: a 9M craft at 8 x 4,096 saturates the RTX PRO 6000's compute alone (863k tok/s);
                                      # four at once ran at ~214k each, their sum one run's rate (2026-10-13): spare VRAM is not spare compute. More than one pays
                                      # only when a single run leaves the card idle, and the proof is the step bars' per-run tok/s summing above one run's rate.
print('decision block:', ' + '.join(f'{k} ({len(g)} keys)' for k, g in GRIDS.items()), '| out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')


In [ ]:
# C2 (P) — PREFLIGHT: the data, each grid's plan, a short run of the hardest composition on this card (the price line), the push probe. No grid training.
import json, os, time, torch
from geolip.nano_beatrix import bed_corpus, bed_runner

# the data: the corpus and the held-out head from fineweb-edu (tokenless; a few minutes; the lines carry the time left)
if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'))

# the plans: what each grid expands to, and what this folder already holds
plans = {}
for name, g in GRIDS.items():
    print('== grid', name)
    plans[name] = bed_runner.run(g, out=OUT, dry=True)

# the price line: the pass's hardest composition (frames to 1,024 bytes, sixteen records a frame) for SMOKE_STEPS on this card
# (the step bar runs; the record stays local under its own short-run name)
smoke = dict(BASE, arms=BASE['arms'].split(',')[0], seeds='1', frame_records='16', sizes='32,128,256,512,1024', steps=SMOKE_STEPS, log_every=20)   # the hardest composition, no continuation
res = bed_runner.run(smoke, out=OUT)
assert res['failed'] is None, 'the short run failed: read the lines above before any grid'
last = [s for s in (bed_runner.parse_step(l) for l in open(f'{OUT}/runner.log', encoding='utf-8')) if s][-1]
per_run = BASE['steps'] * BASE['ctx'] * BASE['mb'] / (last['tok_s'] * 1e3) / 60
todo = sum(p['runs'] - p['done_before'] for p in plans.values())
print(f"this card: {last['tok_s']}k tok/s at {BASE['ctx']} x {BASE['mb']} -> one {BASE['steps']:,}-step run about {per_run:.0f} min alone; "
      f"the {todo} runs left about {per_run * todo / 60:.1f} h one at a time, less with {PARALLEL} at once")

# the push probe (the ship-complete law): a write to the training repo and a read-back BEFORE any training. Tokenless = disarmed.
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = 'bed/preflight/push_probe_pass3.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'tok_s': last['tok_s'], 'grids': GRIDS}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano frame pass 3: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; C3 runs and ships nothing')


In [ ]:
# C3 (T) — THE SESSION: the grids one after the other, PARALLEL runs at once inside each, resume-first (runner_done.txt in OUT); a STOP file
# in OUT starts no new run, lets the running ones finish and skips the grids after; a grid bar over the runs and a step bar per running child fed
# by the trainer's own progress lines; the job-size and gauge lines print through the bars; every finished run ships to REPO in the background
# as it lands: a Hub fault (a 504) retries with a doubling wait and never stops the grid; a record left unshipped is re-shipped at the next
# start and at the end (runner_shipped.txt; the result's 'unshipped' names any that failed). A restart resumes from the records on disk.
# An interrupt loses only the running runs (the bed has no mid-run checkpoint); rerun the cell to continue.
from geolip.nano_beatrix import bed_runner
for name, g in GRIDS.items():
    res = bed_runner.run(g, out=OUT, push=REPO if (HF_TOKEN and PUSH_OK) else '', parallel=PARALLEL)
    print('grid', name, ':', {k: v for k, v in res.items() if k != 'ran'}, '| ran', len(res['ran']))
    if res['stopped']:
        print('STOP seen: the grids after', name, 'are not started; remove the file and rerun the cell to continue')
        break


In [ ]:
# C4 — THE RESULTS: the index over this session's records; the framed rows table carries the text, the bits outside the frames, the data
# frames' body bits by size, the whole value fetched from a frame by its size, the nested whole and the cost. Run any time; the Hub holds the same records.
from IPython.display import Markdown, display
from geolip.nano_beatrix.bed_table import render
display(Markdown(render(f'{OUT}/bed', pairs=True)))


### The other grids

Add, drop or edit an entry of `GRIDS` in C1, rerun C1 (the install is cached), then C3; finished runs are skipped.

| grid | the entry in `GRIDS` |
|---|---|
| the weights source (facts the craft holds in its weights, the question on them) | `weights=dict(BASE, facts='64,256,1024', ask='weights')` — smoked on the 4090, no full run on any card yet: let C2's short run use it first |
| the all-framed diet (task crafts that always see frames) | `plant=1.0` in `BASE`; read the plain-document bits beside the frame gauges, they are the canary |
| the register grid (letters, digits, hex under their own tags) | the first pass: `nano_frame_grid.ipynb` |

A smaller card: `mb=4` at 4,096 (the step count stays; a run takes longer). The session folder is the Colab disk: the Hub copy is the record; a new session rebuilds the corpus in C2 and resumes nothing unless `/content/out/runner_done.txt` is restored.
